# 02 — Three answers

| Primitive | Question | You return |
|---|---|---|
| **Choice** | which label? | a distribution over names |
| **Score** | where on an ordered scale? | a distribution over levels, and their mean |
| **Noul** | is this true? | P(yes) |

**Confidence** (choice and score only) is how peaked that distribution is: `1 - entropy / log(k)`. A one-hot distribution scores 1. A uniform distribution scores 0. Noul does not get a second number; the probability is the answer.

**You implement:** `choice_probs`, `noul_prob`, `confidence`.



In [ ]:
from __future__ import annotations

import torch
from torch import Tensor


def choice_probs(logits: Tensor) -> Tensor:
    raise NotImplementedError("choice_probs")


def noul_prob(logit: Tensor) -> Tensor:
    raise NotImplementedError("noul_prob")


def confidence(probs: Tensor) -> Tensor:
    """probs sums to 1 on the last axis. Return a tensor in [0, 1]."""
    raise NotImplementedError("confidence")



<details>
<summary><b>Solution</b> (try yourself first — click to expand)</summary>

```python
import math
import torch
from torch import Tensor

def choice_probs(logits: Tensor) -> Tensor:
    return torch.softmax(logits.float(), dim=-1)

def noul_prob(logit: Tensor) -> Tensor:
    return torch.sigmoid(logit.float())

def confidence(probs: Tensor) -> Tensor:
    probs = probs.float()
    k = probs.size(-1)
    if k < 2:
        return torch.ones(probs.shape[:-1], device=probs.device, dtype=probs.dtype)
    entropy = -(probs.clamp_min(1e-8) * probs.clamp_min(1e-8).log()).sum(dim=-1)
    return (1.0 - entropy / math.log(k)).clamp(0.0, 1.0)
```

Copy into the stub cell above, then run **Check**.

</details>



## Check



In [ ]:
logits = torch.tensor([0.0, 0.0, 4.0])
probs = choice_probs(logits)
assert torch.allclose(probs.sum(), torch.tensor(1.0), atol=1e-5)
assert int(probs.argmax()) == 2
assert abs(float(noul_prob(torch.tensor(0.0))) - 0.5) < 1e-5
assert float(confidence(torch.tensor([0.0, 1.0]))) > 0.99
assert float(confidence(torch.tensor([0.5, 0.5]))) < 0.01
print("ok", probs)

